# Classes, dataclasses and enums

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python-05-classes-and-dataclasses.ipynb)

You write classes when a group of values and the functions that work on them belong together: a connection with its
settings, a configuration with its validation, a record with its rules. Modern Python gives you three tools for data
records: plain classes, `dataclass` and `Enum`. This chapter shows when to use each.

## A class with state and behaviour

`__init__` sets up a new object. `self` is the object itself. Methods are functions defined in the class and called
on an instance. `__repr__` controls how the object prints, which matters when you debug a pipeline.

In [ ]:
class Batch:
    def __init__(self, name, rows):
        self.name = name
        self.rows = list(rows)

    def __repr__(self):
        return f"Batch(name={self.name!r}, rows={len(self.rows)})"

    def row_count(self):
        return len(self.rows)

batch = Batch("2024-01-15", [1, 2, 3])
assert batch.row_count() == 3
assert repr(batch) == "Batch(name='2024-01-15', rows=3)"

## Class attributes versus instance attributes

An instance attribute belongs to one object. A class attribute is shared by all instances of the class. Assigning
through `self` creates an instance attribute and does not change the class attribute.

In [ ]:
class Pipeline:
    default_retries = 3            # shared by every Pipeline

    def __init__(self, name):
        self.name = name
        self.retries = Pipeline.default_retries

a, b = Pipeline("a"), Pipeline("b")
Pipeline.default_retries = 5
assert a.retries == 3 and b.retries == 3   # copied at construction
assert Pipeline("c").retries == 5

## Properties: validate on assignment

A property looks like an attribute but runs code on read and write. Use it to keep an object valid: reject a bad value
once, at the point of assignment, rather than failing much later.

In [ ]:
class Partition:
    def __init__(self, days):
        self.days = days           # goes through the setter below

    @property
    def days(self):
        return self._days

    @days.setter
    def days(self, value):
        if value < 1:
            raise ValueError("a partition covers at least one day")
        self._days = value

p = Partition(7)
assert p.days == 7
try:
    p.days = 0
except ValueError:
    pass
else:
    raise AssertionError("expected ValueError")

## Inheritance and polymorphism

A subclass reuses its parent's methods and can override them. Code that calls a method on a parent type works with
every subclass: that is **polymorphism**. Prefer small, shallow hierarchies. Composition (one object holding another) is
often clearer than deep inheritance.

In [ ]:
class Source:
    def read(self):
        raise NotImplementedError

class CsvSource(Source):
    def read(self):
        return ["csv row"]

class ApiSource(Source):
    def read(self):
        return ["api row"]

def load_all(sources):
    return [row for source in sources for row in source.read()]

assert load_all([CsvSource(), ApiSource()]) == ["csv row", "api row"]
assert isinstance(CsvSource(), Source)

## Dataclasses: records with less boilerplate

`@dataclass` writes `__init__`, `__repr__` and `__eq__` from the field annotations. Use `frozen=True` for values that
must not change, such as a key or a configuration. Add `slots=True` to save memory when you create millions of them.

In [ ]:
from dataclasses import dataclass, field, asdict, replace

@dataclass(frozen=True, slots=True)
class OrderKey:
    customer_id: int
    order_date: str

@dataclass
class Run:
    job: str
    attempts: int = 0
    tags: list[str] = field(default_factory=list)   # a new list per instance

first = Run("load")
second = Run("load")
first.tags.append("nightly")
assert second.tags == []                  # not shared

key = OrderKey(7, "2024-01-15")
assert key == OrderKey(7, "2024-01-15")
assert asdict(key) == {"customer_id": 7, "order_date": "2024-01-15"}

updated = replace(first, attempts=1)      # a changed copy; the original is untouched
assert updated.attempts == 1 and first.attempts == 0

try:
    key.customer_id = 8
except Exception:
    pass
else:
    raise AssertionError("frozen dataclasses cannot be changed")

## Enums: a fixed set of named values

An `Enum` gives names to a fixed set of values. Comparing against an enum member catches typos that a plain string
would hide: `Status.OK` fails loudly if the member does not exist, but `"ok "` silently does not match `"ok"`.

In [ ]:
from enum import Enum

class Status(str, Enum):
    OK = "ok"
    FAILED = "failed"

def is_done(status):
    return status == Status.OK

assert is_done("ok")                       # str mixin: compares equal to the raw value
assert Status("failed") is Status.FAILED   # look up by value
assert [s.value for s in Status] == ["ok", "failed"]

try:
    Status("ok ")                          # the trailing space is a different value
except ValueError:
    pass
else:
    raise AssertionError("expected ValueError")

## Practice questions

1. Write a `@dataclass(frozen=True)` called `Money` with `amount` (a `Decimal`) and `currency` (a string), and a method
   `add(other)` that refuses to add different currencies.
2. Explain why `Money` should be frozen, in one sentence.
3. Write `class Counter` with `increment()` and a `value` property that starts at 0.

## Answers

### Question 1

In [ ]:
from dataclasses import dataclass
from decimal import Decimal

@dataclass(frozen=True)
class Money:
    amount: Decimal
    currency: str

    def add(self, other):
        if self.currency != other.currency:
            raise ValueError(f"cannot add {other.currency} to {self.currency}")
        return Money(self.amount + other.amount, self.currency)

total = Money(Decimal("10.00"), "EUR").add(Money(Decimal("2.50"), "EUR"))
assert total == Money(Decimal("12.50"), "EUR")

try:
    Money(Decimal("1"), "EUR").add(Money(Decimal("1"), "USD"))
except ValueError:
    pass
else:
    raise AssertionError("expected ValueError")

### Question 2

A money value is a fact. A frozen object cannot be changed by accident, so a total cannot drift after it was computed.

### Question 3

In [ ]:
class Counter:
    def __init__(self):
        self._value = 0

    def increment(self):
        self._value += 1

    @property
    def value(self):
        return self._value

c = Counter()
assert c.value == 0
c.increment(); c.increment()
assert c.value == 2

## Interview notes

- **Use `dataclass` for records, a plain class for behaviour.** A dataclass is a data holder; a class with many methods
  and hidden state is a service.
- **Never use a mutable default in a dataclass field.** Use `field(default_factory=list)`.
- **`frozen=True` gives value semantics.** Equal fields mean equal objects, which makes them safe as dict keys.
- **Prefer composition to deep inheritance.** Two levels of subclassing are usually enough.